In [29]:
%matplotlib qt
#%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
# Use Arial everywhere (falls back if not available)
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 12,          # base font size
    "axes.titlesize": 16,     # figure/axes titles
    "axes.labelsize": 13,     # x/y labels
    "legend.fontsize": 11,    # legend text
    "xtick.labelsize": 11,    # tick labels
    "ytick.labelsize": 11,
    "figure.titlesize": 18,   # suptitle
})

In [30]:
from pathlib import Path
import re

import numpy as np

# Try to use SciPy's griddata for smooth interpolation; fallback to IDW if unavailable.
try:
    from scipy.interpolate import griddata
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# IMPORTANT: channel ordering must match the weight-vector ordering
POS = {
    "AFz": (0.0,  0.75),
    "AF3": (-0.35, 0.70),
    "AF4": ( 0.35, 0.70),

    "Fz":  (0.0,  0.55),
    "F1":  (-0.15, 0.52),
    "F2":  ( 0.15, 0.52),
    "F3":  (-0.30, 0.50),
    "F4":  ( 0.30, 0.50),

    "FC1": (-0.18, 0.32),
    "FC2": ( 0.18, 0.32),
    "FC3": (-0.35, 0.30),
    "FC4": ( 0.35, 0.30),

    "Cz":  (0.0,  0.10),
    "C1":  (-0.18, 0.10),
    "C2":  ( 0.18, 0.10),
    "C3":  (-0.36, 0.08),
    "C4":  ( 0.36, 0.08),
}
FRONTAL_MIDLINE = ['AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4']

# --- Robust parser for weight strings ---
def parse_weight_str(s):
    """
    Parse w_star / w0_lam strings into 1D float array.
    Tries to handle things like:
    - "[0.1 0.2 0.3]"
    - "0.1,0.2,0.3"
    - "0.1;0.2;0.3"
    - "array([0.1, 0.2, 0.3])"
    """
    if s is None:
        return None
    s = str(s).strip()
    if not s:
        return None

    # Remove common wrappers
    # e.g. "array([...])" -> "[...]"
    s = re.sub(r"^array\((.*)\)$", r"\1", s)

    # Strip outer brackets if present
    s = s.strip("[]()")

    # Replace commas and semicolons etc with spaces
    s = s.replace(",", " ").replace(";", " ")

    # Replace anything that is NOT [0-9, sign, dot, exp, space] with space
    # Keeps digits, +/-, dot, e/E, and whitespace
    s_clean = re.sub(r"[^0-9eE+\-\. ]+", " ", s)

    # Collapse multiple spaces
    s_clean = re.sub(r"\s+", " ", s_clean).strip()

    if not s_clean:
        return None

    arr = np.fromstring(s_clean, sep=" ")

    return arr if arr.size > 0 else None

def zscore_per_subject(W):
    W = np.asarray(W, float)
    mean = W.mean(axis=1, keepdims=True)
    std = W.std(axis=1, keepdims=True, ddof=0)
    # avoid division by zero
    std[std == 0] = 1.0
    return (W - mean) / std

def idw_interpolate(points, values, gx, gy, power=2.0, eps=1e-12):
    px = points[:, 0][:, None, None]
    py = points[:, 1][:, None, None]
    dx = gx[None, :, :] - px
    dy = gy[None, :, :] - py
    d2 = dx*dx + dy*dy
    w = 1.0 / (d2 + eps)**(power/2)
    num = (w * values[:, None, None]).sum(axis=0)
    den = w.sum(axis=0)
    return num / (den + eps)

def compute_grid(z, xy, GX, GY, mask, normalise = False):
    z = np.asarray(z, dtype=float)
    if normalise:
        mu = z.mean()
        sd = z.std(ddof=0)
        if sd == 0:
            z_std = np.zeros_like(z)
        else:
            z_std = (z - mu) / sd
        z = z_std  

    if _HAS_SCIPY:
        GZ = griddata(xy, z_std, (GX, GY), method="cubic")
        nan_mask = np.isnan(GZ)
        if nan_mask.any():
            GZ_idw = idw_interpolate(xy, z_std, GX, GY, power=2.0)
            GZ[nan_mask] = GZ_idw[nan_mask]
    else:
        GZ = idw_interpolate(xy, z_std, GX, GY, power=2.0)

    return np.ma.array(GZ, mask=~mask)

def parse_subject_id(val):
    """
    Convert 'subject' column to an integer id.
    Handles 'sub-042', '42', 42, etc.
    """
    if val is None:
        return None
    try:
        # if it's numeric-ish already
        return int(float(val))
    except (ValueError, TypeError):
        s = str(val)
        m = re.search(r"(\d+)", s)
        if not m:
            return None
        return int(m.group(1))

def channel_loadings(X, w):
    """
    Given X (T x C) and weights w (C,), compute
    loadings_j = corr(X[:, j], X @ w) for each channel j.
    """
    X = np.asarray(X, float)
    w = np.asarray(w, float).ravel()
    if X.shape[1] != w.shape[0]:
        raise ValueError(
            f"Dimension mismatch: X has {X.shape[1]} channels but w has length {w.shape[0]}"
        )

    u = X @ w
    u = np.asarray(u).ravel()

    std_u = u.std(ddof=0)
    if std_u == 0:
        return np.full(X.shape[1], np.nan)

    L = np.empty(X.shape[1], dtype=float)
    for j in range(X.shape[1]):
        xj = X[:, j]
        std_xj = xj.std(ddof=0)
        if std_xj == 0:
            L[j] = np.nan
        else:
            L[j] = np.corrcoef(xj, u)[0, 1]
    return L

def normalise_w_cov(X, w, eps=1e-12):
    """
    Normalize weights so that w^T Sxx w = 1, where
    Sxx is the covariance of X (T x C).

    Returns a rescaled copy of w.
    """
    X = np.asarray(X, float)
    w = np.asarray(w, float).ravel()

    # center like np.cov does
    Xc = X - X.mean(axis=0, keepdims=True)
    n = Xc.shape[0]
    if n <= 1:
        return w

    Sxx = (Xc.T @ Xc) / max(n - 1, 1)   # covariance (ddof=1)
    quad = float(w.T @ Sxx @ w)
    if quad <= eps:
        # degenerate case: don't rescale
        return w

    scale = 1.0 / np.sqrt(quad)
    return w * scale


In [48]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# Helpers: draw head + sensors
# ---------------------------
def _draw_head(ax):
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(theta), np.sin(theta), linewidth=2)

    nose_x = np.array([-0.06, 0.0, 0.06])
    nose_y = np.array([1.00, 1.08, 1.00])
    ax.plot(nose_x, nose_y, linewidth=2)

    ear_x = np.array([1.02, 1.07, 1.02])
    ear_y = np.array([0.10, 0.00, -0.10])
    ax.plot( ear_x,  ear_y, linewidth=2)
    ax.plot(-ear_x,  ear_y, linewidth=2)

    ax.set_aspect("equal")
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xticks([])
    ax.set_yticks([])

    # ✅ remove frame/box
    ax.set_axis_off()
    for sp in ax.spines.values():
        sp.set_visible(False)

def plot_empty_channels(POS, channels, title="Channel layout", out_path=None, show=True):
    """
    Plot an "empty scalp map": head outline + channel dots + channel names.
    POS: dict {ch_name: (x,y)} in approx unit-circle coords
    channels: list of channel names to display
    """
    fig, ax = plt.subplots(figsize=(5, 5))
    _draw_head(ax)

    # positions in requested order
    xy = np.array([POS[ch] for ch in channels])
    ax.scatter(xy[:, 0], xy[:, 1], s=80, edgecolors="k", linewidths=1.0)

    # labels
    for ch in channels:
        x, y = POS[ch]
        ax.text(x, y, ch, ha="center", va="bottom", fontsize=9)

    ax.set_title(title)

    if out_path is not None:
        fig.savefig(out_path, dpi=200, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)


# --------------------------------------------------------
# 1) Empty scalp map with ONLY the 17 frontal-midline chans
# --------------------------------------------------------
# Example usage:
# plot_empty_channels(POS, FRONTAL_MIDLINE,
#                     title="17 frontal(-midline) channels",
#                     out_path="layout_17.png")


# --------------------------------------------------------
# 2) 64-channel map with the 17 highlighted
#    Option A: use MNE montage to get 64 positions automatically
# --------------------------------------------------------
def get_pos_from_mne_layout(montage_name="biosemi64"):
    import numpy as np
    import mne

    montage = mne.channels.make_standard_montage(montage_name)

    info = mne.create_info(ch_names=montage.ch_names, sfreq=1000.0, ch_types="eeg")
    info.set_montage(montage)

    layout = mne.channels.find_layout(info)  # public API
    # layout.pos is (n_ch, 4): x, y, width, height (0..1-ish coords)
    pos = layout.pos[:, :2]

    chs = layout.names
    POS_2D = {ch: (float(pos[i, 0]), float(pos[i, 1])) for i, ch in enumerate(chs)}

    # rescale to roughly [-1, 1] for your head outline
    xy = np.array(list(POS_2D.values()))
    xy = xy - xy.mean(axis=0, keepdims=True)
    xy = xy / (np.abs(xy).max() + 1e-12)
    POS_2D = {ch: (float(xy[i, 0]), float(xy[i, 1])) for i, ch in enumerate(chs)}
    return POS_2D



def plot_64_with_highlight(POS_64, highlight_channels,
                           title="64-channel layout (highlighted subset)",
                           out_path=None, show=True,
                           label_all=False, label_highlight=True):
    """
    POS_64: dict {ch: (x,y)} for the 64 electrodes
    highlight_channels: list of channels to highlight (e.g., your 17)
    label_all: if True, labels every channel (will be cluttered)
    label_highlight: label only the highlighted ones
    """
    fig, ax = plt.subplots(figsize=(6, 6))
    _draw_head(ax)

    all_ch = list(POS_64.keys())
    xy_all = np.array([POS_64[ch] for ch in all_ch], dtype=float)

    # base: all sensors
    ax.scatter(xy_all[:, 0], xy_all[:, 1], s=30, edgecolors="k", linewidths=0.8, alpha=0.8)

    # highlight subset (only those that exist in POS_64)
    hi = [ch for ch in highlight_channels if ch in POS_64]
    xy_hi = np.array([POS_64[ch] for ch in hi], dtype=float)
    if len(hi) > 0:
        ax.scatter(xy_hi[:, 0], xy_hi[:, 1], s=90, edgecolors="k", linewidths=1.2)

    # labels
    if label_all:
        for ch in all_ch:
            x, y = POS_64[ch]
            ax.text(x, y, ch, ha="center", va="bottom", fontsize=7)

    if label_highlight:
        for ch in hi:
            x, y = POS_64[ch]
            ax.text(x, y, ch, ha="center", va="bottom", fontsize=11, fontweight="bold", 
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.6, pad=0.2))

    ax.set_title(title)

    if out_path is not None:
        fig.savefig(out_path, dpi=200, bbox_inches=None)

    if show:
        plt.show()
    else:
        plt.close(fig)


# -------------------------
# Example usage for figure 2
# -------------------------
POS_64 = get_pos_from_mne_layout("biosemi64")
plot_64_with_highlight(POS_64, FRONTAL_MIDLINE,
                       title=None,
                       label_all=False, label_highlight=True)


In [ ]:

# --- Configuration ---
CSV_PATH = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\ridge_reg\2seconds_window_ALL\dfk_ridge_ALL.csv")
OUT_PATH = CSV_PATH.with_name("dfk_ridge_TMP_mean_scalpmaps.png")


# --- Load dfk_ridge_TMP.csv and accumulate weights ---
W_star_list = []
W_lam0_list = []

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for i, row in enumerate(reader):
        ws = parse_weight_str(row.get("w_star"))
        wl = parse_weight_str(row.get("w_lam0") or row.get("w0_lam"))

        # Debug: show first few parsed shapes
        if i < 3:
            print(f"Row {i}:")
            print("  raw w_star:", row.get("w_star"))
            print("  parsed w_star shape:", None if ws is None else ws.shape)
            print("  raw w0_lam/w_lam0:", row.get("w0_lam") or row.get("w_lam0"))
            print("  parsed w_lam0 shape:", None if wl is None else wl.shape)

        if ws is None or wl is None:
            continue

        W_star_list.append(ws)
        W_lam0_list.append(wl)

if not W_star_list:
    raise RuntimeError("No valid weight vectors found in w_star / w0_lam columns.")

# Stack
W_star = np.vstack(W_star_list)
W_lam0 = np.vstack(W_lam0_list)

print("W_star shape:", W_star.shape)
print("W_lam0 shape:", W_lam0.shape)

n_subj, n_chan = W_star.shape

if n_chan != len(FRONTAL_MIDLINE):
    raise RuntimeError(
        f"Number of channels in weights ({n_chan}) != number in POS ({len(FRONTAL_MIDLINE)}).\n"
        f"- If {n_chan} is correct, update POS/CHANNELS to have {n_chan} entries\n"
        f"- If you expected more channels, check how w_star / w0_lam are stored in the CSV."
    )

print(f"Loaded {n_subj} subjects, {n_chan} channels.")

# --- Compute mean weights across subjects ---
#mean_star = W_star.mean(axis=0)
#mean_lam0 = W_lam0.mean(axis=0)
# W_star, W_lam0: shape (n_subj, n_chan)

W_star_zsubj = zscore_per_subject(W_star)
W_lam0_zsubj = zscore_per_subject(W_lam0)

# Now average the standardized maps
mean_star = W_star_zsubj.mean(axis=0)   # (n_chan,)
mean_lam0 = W_lam0_zsubj.mean(axis=0)

# --- Geometry / grid for topomaps ---
xy = np.array([POS[ch] for ch in FRONTAL_MIDLINE])

N = 200
grid_x = np.linspace(-1.0, 1.0, N)
grid_y = np.linspace(-1.0, 1.0, N)
GX, GY = np.meshgrid(grid_x, grid_y)
mask = (GX**2 + GY**2) <= 1.0

GZ_star = compute_grid(mean_star, xy, GX, GY, mask, normalise = True)
GZ_lam0 = compute_grid(mean_lam0, xy, GX, GY, mask, normalise = True)

# --- Plot side-by-side scalp maps ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, GZ_masked, title in zip(
    axes,
    [GZ_star, GZ_lam0],
    ["Mean w_star (λ*)", "Mean w0_lam (λ = 0)"]
):
    im = ax.imshow(
        GZ_masked,
        origin="lower",
        extent=(-1, 1, -1, 1),
        interpolation="bilinear",
    )

    # head outline
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(theta), np.sin(theta), linewidth=2)

    # nose
    nose_x = np.array([-0.06, 0.0, 0.06])
    nose_y = np.array([1.00, 1.08, 1.00])
    ax.plot(nose_x, nose_y, linewidth=2)

    # ears
    ear_x = np.array([1.02, 1.07, 1.02])
    ear_y = np.array([0.10, 0.00, -0.10])
    ax.plot( ear_x,  ear_y, linewidth=2)
    ax.plot(-ear_x,  ear_y, linewidth=2)

    # electrodes + labels
    ex, ey = xy[:, 0], xy[:, 1]
    ax.scatter(ex, ey, s=25, edgecolors="k")
    for ch, (cx, cy) in POS.items():
        ax.text(cx, cy, ch, ha="center", va="bottom", fontsize=8)

    ax.set_aspect("equal")
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)

fig.subplots_adjust(right=0.88)

# [left, bottom, width, height] in figure coordinates
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Mean normalized weight (wᵀSxxw = 1)")
fig.tight_layout(rect=[0, 0, 0.88, 0.96])  # match the right margin with subplots_adjust
fig.savefig(OUT_PATH, dpi=200)
print(f"Saved scalp maps to: {OUT_PATH}")
plt.show()


Row 0:
  raw w_star: [[ 0.02244426]
 [ 0.00308325]
 [ 0.0033981 ]
 [ 0.00258686]
 [ 0.0037116 ]
 [ 0.00326252]
 [ 0.00488914]
 [ 0.00466294]
 [ 0.0206698 ]
 [ 0.00385643]
 [ 0.0060068 ]
 [ 0.0003146 ]
 [ 0.00185774]
 [-0.00845835]
 [-0.00021064]
 [ 0.00822715]
 [ 0.00600372]]
  parsed w_star shape: (17,)
  raw w0_lam/w_lam0: [[ 0.04508407]
 [-0.0023489 ]
 [-0.00632297]
 [-0.00413349]
 [ 0.01307881]
 [-0.00551107]
 [-0.00124359]
 [ 0.01658223]
 [ 0.04263676]
 [-0.00040891]
 [ 0.02020082]
 [-0.01416511]
 [-0.00217884]
 [-0.02760512]
 [ 0.00378629]
 [ 0.01911352]
 [ 0.00903516]]
  parsed w_lam0 shape: (17,)
Row 1:
  raw w_star: [[ 0.02955352]
 [ 0.01185699]
 [ 0.02701653]
 [ 0.03078019]
 [-0.02812053]
 [-0.00865676]
 [ 0.02862904]
 [-0.03033125]
 [ 0.00590562]
 [ 0.01428876]
 [ 0.00053917]
 [ 0.03285075]
 [-0.01880903]
 [ 0.00439743]
 [ 0.02489583]
 [ 0.00288526]
 [ 0.01619639]]
  parsed w_star shape: (17,)
  raw w0_lam/w_lam0: [[ 0.02820643]
 [ 0.01401407]
 [ 0.0375922 ]
 [ 0.0546998 ]
 

C:\Users\cdd\AppData\Local\Temp\ipykernel_8040\1506774374.py:132: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 0.88, 0.96])  # match the right margin with subplots_adjust


Saved scalp maps to: C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\ridge_reg\2seconds_window_ALL\dfk_ridge_TMP_mean_scalpmaps.png


In [25]:
import csv
import re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.interpolate import griddata
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# -------------------------
# Electrode layout
# -------------------------
POS = {
    "AFz": (0.0,  0.75),
    "AF3": (-0.35, 0.70),
    "AF4": ( 0.35, 0.70),
    "Fz":  (0.0,  0.55),
    "F1":  (-0.15, 0.52),
    "F2":  ( 0.15, 0.52),
    "F3":  (-0.30, 0.50),
    "F4":  ( 0.30, 0.50),
    "FC1": (-0.18, 0.32),
    "FC2": ( 0.18, 0.32),
    "FC3": (-0.35, 0.30),
    "FC4": ( 0.35, 0.30),
    "Cz":  (0.0,  0.10),
    "C1":  (-0.18, 0.10),
    "C2":  ( 0.18, 0.10),
    "C3":  (-0.36, 0.08),
    "C4":  ( 0.36, 0.08),
}
FRONTAL_MIDLINE = ['AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4']

# -------------------------
# Helpers
# -------------------------
def parse_weight_str(s):
    if s is None:
        return None
    s = str(s).strip()
    if not s:
        return None
    s = re.sub(r"^array\((.*)\)$", r"\1", s)
    s = s.strip("[]()")
    s = s.replace(",", " ").replace(";", " ")
    s_clean = re.sub(r"[^0-9eE+\-\. ]+", " ", s)
    s_clean = re.sub(r"\s+", " ", s_clean).strip()
    if not s_clean:
        return None
    arr = np.fromstring(s_clean, sep=" ")
    return arr if arr.size > 0 else None

def idw_interpolate(points, values, gx, gy, power=2.0, eps=1e-12):
    px = points[:, 0][:, None, None]
    py = points[:, 1][:, None, None]
    dx = gx[None, :, :] - px
    dy = gy[None, :, :] - py
    d2 = dx*dx + dy*dy
    w = 1.0 / (d2 + eps)**(power/2)
    num = (w * values[:, None, None]).sum(axis=0)
    den = w.sum(axis=0)
    return num / (den + eps)

def compute_grid(z, xy, GX, GY, mask, normalise = False):
    z = np.asarray(z, dtype=float)
    if normalise:
        mu = z.mean()
        sd = z.std(ddof=0)
        if sd == 0:
            z_std = np.zeros_like(z)
        else:
            z_std = (z - mu) / sd
        z = z_std  

    if _HAS_SCIPY:
        GZ = griddata(xy, z_std, (GX, GY), method="cubic")
        nan_mask = np.isnan(GZ)
        if nan_mask.any():
            GZ_idw = idw_interpolate(xy, z_std, GX, GY, power=2.0)
            GZ[nan_mask] = GZ_idw[nan_mask]
    else:
        GZ = idw_interpolate(xy, z_std, GX, GY, power=2.0)

    return np.ma.array(GZ, mask=~mask)

def sign_align(W):
    """Flip subjects so they align with the group mean (prevents cancellation)."""
    W = np.asarray(W, float).copy()
    ref = W.mean(axis=0)
    for i in range(W.shape[0]):
        if np.dot(W[i], ref) < 0:
            W[i] *= -1
    return W

def load_W_pair(csv_path):
    W_star_list, W_lam0_list = [], []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            ws = parse_weight_str(row.get("w_star"))
            wl = parse_weight_str(row.get("w_lam0"))
            if ws is None or wl is None:
                continue
            W_star_list.append(ws)
            W_lam0_list.append(wl)

    if not W_star_list:
        raise RuntimeError(f"No valid weights in: {csv_path}")

    W_star = np.vstack(W_star_list)
    W_lam0 = np.vstack(W_lam0_list)

    if W_star.shape[1] != len(FRONTAL_MIDLINE):
        raise RuntimeError(f"{csv_path}: weights length {W_star.shape[1]} != {len(FRONTAL_MIDLINE)}")

    return W_star, W_lam0

def draw_topomap(ax, GZ, xy, vlim):
    im = ax.imshow(
        GZ, origin="lower", extent=(-1, 1, -1, 1),
        interpolation="bilinear",
        vmin=-vlim, vmax=vlim
    )

    # head outline
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(theta), np.sin(theta), lw=2)

    # nose
    ax.plot([-0.06, 0.0, 0.06], [1.00, 1.08, 1.00], lw=2)

    # ears
    ear_x = np.array([1.02, 1.07, 1.02])
    ear_y = np.array([0.10, 0.00, -0.10])
    ax.plot( ear_x, ear_y, lw=2)
    ax.plot(-ear_x, ear_y, lw=2)

    # electrodes (dots only — no text clutter)
    ax.scatter(xy[:, 0], xy[:, 1], s=28, facecolors="none", edgecolors="k", lw=1)

    ax.set_aspect("equal")
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

    return im

# -------------------------
# Paths
# -------------------------
ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\ridge_reg")
SETUPS = [
    ("Combined", ROOT / "2seconds_window_ALL"  / "dfk_ridge_ALL.csv"),
    ("Tonic",    ROOT / "2seconds_window_SLOW" / "dfk_ridge.csv"),
    ("Phasic",   ROOT / "1seconds_window_FAST" / "dfk_ridge.csv"),
]

# -------------------------
# Grid geometry
# -------------------------
xy = np.array([POS[ch] for ch in FRONTAL_MIDLINE])
N = 220
grid_x = np.linspace(-1.0, 1.0, N)
grid_y = np.linspace(-1.0, 1.0, N)
GX, GY = np.meshgrid(grid_x, grid_y)
mask = (GX**2 + GY**2) <= 1.0

# -------------------------
# Compute all maps first (amplitude-preserving)
# -------------------------
maps = []
all_vals = []

for name, csv_path in SETUPS:
    W_star, W_lam0 = load_W_pair(csv_path)

    # Optional but recommended: align sign to avoid cancellation
    W_star = sign_align(W_star)
    W_lam0 = sign_align(W_lam0)

    mean_star = zscore_per_subject(W_star).mean(axis=0)
    mean_lam0 = zscore_per_subject(W_lam0).mean(axis=0)

    GZ_star = compute_grid(mean_star, xy, GX, GY, mask, normalise=True)
    GZ_lam0 = compute_grid(mean_lam0, xy, GX, GY, mask, normalise=True)


    maps.append((name, GZ_star, GZ_lam0))
    all_vals.append(GZ_star.compressed())
    all_vals.append(GZ_lam0.compressed())

all_vals = np.concatenate(all_vals)
vlim = np.nanpercentile(np.abs(all_vals), 99)  # robust symmetric scaling

# -------------------------
# Plot: 3x2 + dedicated colorbar column
# -------------------------
fig = plt.figure(figsize=(9.0, 10.5))
gs = fig.add_gridspec(nrows=3, ncols=3, width_ratios=[1, 1, 0.06], wspace=0.15, hspace=0.22)

axes = [[fig.add_subplot(gs[r, c]) for c in range(2)] for r in range(3)]
cax = fig.add_subplot(gs[:, 2])

# Column headers
axes[0][0].set_title(r"$\lambda_s^\ast$", fontsize=14, pad=10)
axes[0][1].set_title(r"$\lambda = 0$", fontsize=14, pad=10)

# Draw maps
last_im = None
for r, (name, GZ_star, GZ_lam0) in enumerate(maps):
    last_im = draw_topomap(axes[r][0], GZ_star, xy, vlim=vlim)
    draw_topomap(axes[r][1], GZ_lam0, xy, vlim=vlim)

    # Row label (clean, vertical, big)
    axes[r][0].text(-0.32, 0.5, name, transform=axes[r][0].transAxes,
                    rotation=90, ha="center", va="center",
                    fontsize=14, fontweight="bold")

# Shared colorbar
cb = fig.colorbar(last_im, cax=cax)
cb.set_label("Mean ridge weight (a.u.)", rotation=90)

# Optional: overall title
# fig.suptitle("Mean ridge weights (frontal midline ROI)", y=0.98)

fig.savefig(ROOT / "scalpmaps_mean_weights_ALL_components_clean.png", dpi=300, bbox_inches="tight")
plt.show()


In [28]:
# -------------------------
# Plot: 2x3 + dedicated colorbar column (TRANSPOSED)
# rows:   λ* (top), λ=0 (bottom)
# cols:   Combined, Tonic, Phasic
# -------------------------
fig = plt.figure(figsize=(12.0, 6.8))
gs = fig.add_gridspec(
    nrows=2, ncols=4,
    width_ratios=[1, 1, 1, 0.06],
    wspace=0.15, hspace=0.18
)

axes = [[fig.add_subplot(gs[r, c]) for c in range(3)] for r in range(2)]
cax = fig.add_subplot(gs[:, 3])

# Column headers (components)
for c, (name, _, _) in enumerate(maps):
    axes[0][c].set_title(name, fontsize=14, pad=10)

# Row labels
axes[0][0].text(-0.22, 0.5, r"$\lambda = 0$", transform=axes[0][0].transAxes,
                rotation=90, ha="center", va="center", fontsize=14, fontweight="bold")
axes[1][0].text(-0.22, 0.5, r"$\lambda_s^\ast$", transform=axes[1][0].transAxes,
                rotation=90, ha="center", va="center", fontsize=14, fontweight="bold")

# Draw maps: top row = λ=0, bottom row = λ*
for c, (name, GZ_star, GZ_lam0) in enumerate(maps):
    last_im = draw_topomap(axes[0][c], GZ_lam0, xy, vlim=vlim)  # λ=0 on top
    draw_topomap(axes[1][c], GZ_star, xy, vlim=vlim)           # λ* on bottom


# Shared colorbar
cb = fig.colorbar(last_im, cax=cax)
cb.set_label("Mean ridge weight (a.u.)", rotation=90)

fig.savefig(ROOT / "scalpmaps_mean_weights_ALL_components_clean_2x3.png",
            dpi=300, bbox_inches="tight")
plt.show()
